# 18 · UDFs: when (not) to use them (Spark)

A **UDF** (user-defined function) runs your own Python code on each value.

- A plain **Python UDF** sends every row from the JVM (where Spark runs) to a Python process and back:
  slow, and Spark cannot optimize inside it.
- A **pandas UDF** (vectorized) sends whole batches with Apache Arrow: much faster.
- **Built-in functions** (`F.*`) are fastest: always try them first.

Interview answer: "I avoid Python UDFs; I use built-ins, then pandas UDFs if I really need Python."

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/18_udfs.ipynb`](../../spark/practice/18_udfs.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Python UDF vs built-in  ·  🟢 easy

Make a column `name_upper` two ways: a Python UDF and `F.upper`. Show both on 3 rows and print both plans. Find `BatchEvalPython` in the UDF plan.

**How it works:** The UDF runs Python per row (BatchEvalPython in the plan); `F.upper` runs inside the JVM. Same output.

In [ ]:
@F.udf(returnType="string")
def to_upper(s):
    return s.upper() if s is not None else None

employees.select("name", to_upper("name").alias("name_upper")).show(3)
employees.select("name", to_upper("name").alias("u")).explain()
employees.select("name", F.upper("name").alias("u")).explain()

+-------------+-------------+
|         name|   name_upper|
+-------------+-------------+
|Alice Johnson|ALICE JOHNSON|
|    Bob Smith|    BOB SMITH|
|  Carol White|  CAROL WHITE|
+-------------+-------------+
only showing top 3 rows
== Physical Plan ==
*(1) Project [name#3, pythonUDF0#1929 AS u#1928]
+- BatchEvalPython [to_upper(name#3)#1927], [pythonUDF0#1929]
   +- FileScan csv [name#3] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:<repo>/data/employees.tsv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<name:string>


== Physical Plan ==
*(1) Project [name#3, upper(name#3) AS u#1930]
+- FileScan csv [name#3] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:<repo>/data/employees.tsv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<name:string>

**Why / traps:** `BatchEvalPython` = rows leave the JVM. The built-in stays in one optimized `Project` step (and gets whole-stage code generation).

## Q2. Handle NULL inside a UDF  ·  🟢 easy

Write a UDF `city_code` that returns the first 3 letters of a city in capitals. Run it on `customers` **without** a NULL check first (it fails on the NULL city), then fix it.

*Expected result: 2 rows · columns: `customer_id`, `city`, `code`*

**How it works:** The first UDF crashes on `None`; the fixed one returns None when the input is None.

In [ ]:
bad = F.udf(lambda c: c[:3].upper(), "string")
try:
    customers.select(bad("city")).collect()
except Exception as e:
    print("fails:", type(e).__name__)

city_code = F.udf(lambda c: c[:3].upper() if c else None, "string")
res = customers.filter("customer_id IN (1, 13)").select("customer_id", "city", city_code("city").alias("code"))
res.show()

fails: PythonException

+-----------+-------+----+
|customer_id|   city|code|
+-----------+-------+----+
|          1|Chicago| CHI|
|         13|   NULL|NULL|
+-----------+-------+----+

**Why / traps:** Python gets NULL as `None`. `None[:3]` raises an error and the **whole job** fails. Built-ins handle NULL for you.

## Q3. Wrong return type gives silent NULLs  ·  🟡 medium

Declare a UDF that returns `int` but really returns a float (salary / 1000). What do you get? Fix the declared type.

*Expected result: 2 rows · columns: `salary`, `k`*

**How it works:** Spark can't convert the float into the declared int, so it returns NULL. Declaring `double` fixes it.

In [ ]:
wrong = F.udf(lambda s: float(s) / 1000, "int")
employees.select("salary", wrong("salary").alias("k")).show(2)

right = F.udf(lambda s: float(s) / 1000, "double")
res = employees.filter("emp_id <= 2").select("salary", right("salary").alias("k"))
res.show()

+---------+----+
|   salary|   k|
+---------+----+
|250000.00|NULL|
|150000.00|NULL|
+---------+----+
only showing top 2 rows

+---------+-----+
|   salary|    k|
+---------+-----+
|250000.00|250.0|
|150000.00|150.0|
+---------+-----+

**Why / traps:** A Python UDF whose value does not match the declared type returns **NULL**, with no error. Easy to miss in a pipeline: add a NULL-count check after UDF columns.

## Q4. Pandas UDF (vectorized)  ·  🟡 medium

Write a pandas UDF `with_tax` that multiplies a price Series by 1.18 and rounds to 2 decimals. Apply it to `products`.

*Expected result: 5 rows · columns: `product_name`, `price`, `price_with_tax`*

**How it works:** A pandas UDF gets a batch of values as a pandas Series and returns a Series of the same length.

In [ ]:
import pandas as pd

@F.pandas_udf("double")
def with_tax(price: pd.Series) -> pd.Series:
    return (price.astype(float) * 1.18).round(2)

res = products.select("product_name", "price", with_tax("price").alias("price_with_tax")).limit(5)
res.show()

+-----------------+-----+--------------+
|     product_name|price|price_with_tax|
+-----------------+-----+--------------+
|   Wireless Mouse|25.00|          29.5|
|      USB-C Cable|12.00|         14.16|
|Bluetooth Speaker|60.00|          70.8|
|     Laptop Stand|45.00|          53.1|
|           Webcam|60.00|          70.8|
+-----------------+-----+--------------+

**Why / traps:** The function gets a whole **pandas Series** (a batch of rows), not one value. Needs `pyarrow`. The built-in `F.round(F.col("price") * 1.18, 2)` is still faster.

## Q5. Use a UDF in Spark SQL  ·  🟢 easy

Register a Python function as `initials` for SQL with `spark.udf.register`, then call it in `%%sql`.

*Expected result: 3 rows · columns: `name`, `ini`*

**How it works:** `spark.udf.register(name, function, type)` makes the Python function callable from SQL.

In [ ]:
def initials(name):
    return "".join(p[0] for p in name.split()) if name else None

spark.udf.register("initials", initials, "string")
res = spark.sql("SELECT name, initials(name) AS ini FROM employees WHERE emp_id <= 3")
res.show()

+-------------+---+
|         name|ini|
+-------------+---+
|Alice Johnson| AJ|
|    Bob Smith| BS|
|  Carol White| CW|
+-------------+---+

**Spark SQL:**

In [ ]:
%%sql
SELECT name, initials(name) AS ini FROM employees WHERE emp_id <= 3

,name,ini
0,Alice Johnson,AJ
1,Bob Smith,BS
2,Carol White,CW


**Why / traps:** Run the Python cell first: it registers the function for this Spark session.

## Q6. Replace a UDF with built-ins  ·  🟡 medium

This UDF labels orders: amount NULL → `missing`, > 200 → `big`, else `normal`. Write the same with `F.when` and check that both give the same counts per label.

*Expected result: 3 rows · columns: `label`, `count`*

**How it works:** `when` / `otherwise` builds the same labels without Python; both give the same counts.

In [ ]:
%%sql
SELECT CASE WHEN amount IS NULL THEN 'missing'
            WHEN amount > 200 THEN 'big'
            ELSE 'normal' END AS label,
       COUNT(*) AS count
FROM orders
GROUP BY 1

,label,count
0,normal,93
1,missing,6
2,big,9


**DataFrame API:**

In [ ]:
label_udf = F.udf(lambda a: "missing" if a is None else ("big" if a > 200 else "normal"), "string")
by_udf = orders.groupBy(label_udf("amount").alias("label")).count()

label = (F.when(F.col("amount").isNull(), "missing")
          .when(F.col("amount") > 200, "big")
          .otherwise("normal"))
res = orders.groupBy(label.alias("label")).count()
print("same result:", sorted(by_udf.collect()) == sorted(res.collect()))
res.show()

same result: True

+-------+-----+
|  label|count|
+-------+-----+
| normal|   93|
|missing|    6|
|    big|    9|
+-------+-----+

**Why / traps:** Most UDFs in real code can be replaced by `when`, `regexp_extract`, `split`, `transform`, or a join to a lookup table.

## Q7. Per-group pandas logic: applyInPandas  ·  🔴 hard

For each department, use pandas to compute each employee's salary **z-score** ((salary − mean) / std) with `groupBy(...).applyInPandas`. Skip NULL dept_id.

*Expected result: 29 rows · columns: `dept_id`, `name`, `z`*

**How it works:** `applyInPandas` sends each department's rows to Python as one pandas DataFrame; the function returns a DataFrame with the given schema.

In [ ]:
import pandas as pd

def zscore(pdf: pd.DataFrame) -> pd.DataFrame:
    s = pdf["salary"].astype(float)
    std = s.std(ddof=0)
    pdf["z"] = ((s - s.mean()) / std).round(2) if std else 0.0
    return pdf[["dept_id", "name", "z"]]

res = (employees.filter(F.col("dept_id").isNotNull())
       .groupBy("dept_id").applyInPandas(zscore, schema="dept_id int, name string, z double")
       .orderBy("dept_id", F.desc("z")))
res.show()

+-------+-------------+-----+
|dept_id|         name|    z|
+-------+-------------+-----+
|      1|Alice Johnson|  2.3|
|      1|    Bob Smith| 0.31|
|      1|  Carol White|-0.28|
|      1|  David Brown|-0.28|
|      1|    Eva Green|-0.48|
|      1|  Frank Moore|-0.78|
|      1|    Grace Lee|-0.78|
|      2|  Henry Clark| 2.03|
|      2|  Irene Lewis|-0.03|
|      2|  Jack Walker|-0.03|
|      2|   Karen Hall|-0.03|
|      2|    Leo Allen|-0.86|
|      2|    Mia Young|-1.07|
|      3|  Nathan King| 1.58|
|      3| Olivia Scott| -0.2|
|      3|   Paul Adams| -0.2|
|      3|  Quinn Baker|-1.19|
|      4|Rachel Nelson| 1.73|
|      4|   Sam Carter|-0.49|
|      4|Tina Mitchell|-0.62|
+-------+-------------+-----+
only showing top 20 rows

**Why / traps:** Each group goes to Python as one pandas DataFrame, so **one group must fit in memory**. A skewed key can break it. The same z-score with window functions: `(salary - avg(salary) over w) / stddev_pop(salary) over w`.